# <div style="text-align: center"> Introduction </div>

Along these tutorials, we will see how <span style="color:blue">**SCOPE**</span> interacts with the different parts of the code interact to handle the execution of computational workflows. 

These are the topics covered in each tutorial:
1) The **System**, **Specie**, **Cell** and **Atom** classes  
2) The Computational workflow: **Branch**, **Workflow**, **Job**, and **Computation** classes  
3) The **State** class  
4) The **Data**, **Collection** and **VNM** classes
5) The **Input_data** class, and **scope input files**

# <div style="text-align: center"> Tutorial 5</div>

In this tutorial, we will have a look at an input file of Scope. The input is how we indicate Scope to interact with a System and perform a task.  
**Scope Input Files** are interpreted by Scope through the **Input_data** class


In [1]:
import os
import scope

In [2]:
## Path of the data folder. It should be "os.path.abspath('.')+'/Data/Tutorial_4/"
tutorial_folder = os.path.abspath('.')+'/Data/Tutorial_5/'

## 1) Create an Input File

In [3]:
from scope.classes_input import *

In [4]:
## Input Files can be created from a file, or from a string. 
## This is an example input, read as a string. We will soon go through the different parts
## First, notice that there are four sections, initiated with '&', and terminated with '/'.
## Sections can be empty as in the example below (see &options) or even absent. In those cases, default values will be added
## Also, notice that the values are either introduced as strings 'X', lists [], booleans, or integers/floats...
## Please respect the notation used for each type of variable

test_input = """
&environment
   max_jobs         = 80
   max_procs        = 240
   requested_procs  = 8
/

&options
/

&job_data
   branch       = 'Isolated'
   workflow     = ['reference']
   job_setup    = 'regular'
   suffix       = 'opt'
   keyword      = 'pbe opt'

   istate       = 'initial'
   fstate       = 'pbe opt'

   hierarchy    = 1
   requisites   = []
   constrains   = ['self']
   must_be_good = False
/

&qc_data
   software     = 'g16'
   jobtype      = 'opt'
   functional   = 'pbe'
   basis        = 'def2SVP'
   is_grimme    = True
   loose_opt    = True
/"""

In [5]:
# IMPORTANT. Notice that, when running the 'scope_run_input' script. The input file (-i arg) must be an actual file, not a string as in the cell above
# Thus, we save this text to an actual file, so we can call it later
from scope.read_write import save_text
file_name = ''.join([tutorial_folder,"test_input.inp"])
save_text(test_input, file_name)

In [6]:
## In any case, the input is interpreted and stored as INPUT_DATA class objects:
## There are 4 input sections. Which are read separately 
local_env   = Input_data(content=test_input, section="&environment", isfile=False)
options     = Input_data(content=test_input, section="&options", isfile=False)
job_data    = Input_data(content=test_input, section="&job_data", isfile=False)
qc_data     = Input_data(content=test_input, section="&qc_data", isfile=False)

## They could be read in a single line (see below), but the data would not be stored correctly 
## inp = input_data(content=test_input, isfile=False)  

INPUT_DATA.READ: Section '&options' is empty. Only defaults will be taken


### Section 1: Local Environment

In [7]:
# Here are the options related to the computation resources, that are not hardcoded during the environment creation. 
# The user can change these options anytime in the job file, without having to modify the environment
local_env

Formatted input interpretation: ( self -> Instance of class Input() )
---------------------------------------------------
self.Key                 | Data Type           | Value     
---------------------------------------------------
self.max_jobs            | <class 'int'>       | 80        
self.max_procs           | <class 'int'>       | 240       
self.requested_procs     | <class 'int'>       | 8         

In [8]:
# There are some defaults as well, which are always added to the user options
local_env = fill_environment_data(local_env)
print(local_env)

Formatted input interpretation: ( self -> Instance of class Input() )
---------------------------------------------------
self.Key                 | Data Type           | Value     
---------------------------------------------------
self.max_jobs            | <class 'int'>       | 80        
self.max_procs           | <class 'int'>       | 240       
self.requested_procs     | <class 'int'>       | 8         
self.method              | <class 'str'>       | weighted  



In [22]:
## Notice that a "method" variable has been added as a default. This one specifies how best partition/queue is chosen if more than one is available

### Section 2: Options

In [10]:
## These are other options related to the submission:
options = fill_options_data(options)
print(options)

## This section was empty in the "test_input", but now defaults were added:

# want_submit:  
    # Here, the user can turn off submission (want_submit = False) of jobs. 
    # If so, input files will be created, but computations won't be submitted. 
    # This is perfect for testing the input files creation

# ignore_submitted: 
    # Normally when submitting a computation, SCOPE verifies that no computation with the same name is already running. 
    # If activated (ignore_submitted = True), this check won't be done, and multiple instances of the same job will be created.
    # This might be OK in some cases, when those job, even if they have the same name, refer to different input files.

# overwrite_inputs and overwrite_logs:
    # When true, the code can overwrite files that already exist.

Formatted input interpretation: ( self -> Instance of class Input() )
---------------------------------------------------
self.Key                 | Data Type           | Value     
---------------------------------------------------
self.want_submit         | <class 'bool'>      | True      
self.ignore_submitted    | <class 'bool'>      | False     
self.overwrite_inputs    | <class 'bool'>      | True      
self.overwrite_outputs   | <class 'bool'>      | True      



### Section 3: Job Data Section

In [23]:
## This section includes job information. 
# 
## Basically, a Job associated with this information will be created (if it doesn't exist yet) or found (if it already exists) inside each system
job_data = fill_job_data(job_data)
print(job_data)

Formatted input interpretation: ( self -> Instance of class Input() )
---------------------------------------------------
self.Key                 | Data Type           | Value     
---------------------------------------------------
self.branch              | <class 'str'>       | Isolated  
self.workflow            | <class 'list'>      | ['reference']
self.job_setup           | <class 'str'>       | regular   
self.suffix              | <class 'str'>       | opt       
self.keyword             | <class 'str'>       | pbe_opt   
self.istate              | <class 'str'>       | initial   
self.fstate              | <class 'str'>       | pbe_opt   
self.hierarchy           | <class 'int'>       | 1         
self.requisites          | <class 'list'>      | []        
self.constrains          | <class 'list'>      | ['self']  
self.must_be_good        | <class 'bool'>      | False     



In [12]:
## Here's the meaning of each variable:

# branch:       to locate the job inside the system, the BRANCH must be specified. 
# workflow:     to locate the job inside the system, the WORKFLOW must be specified. If more than one, one job will be created inside each workflow
# setup:        how are COMPUTATIONS inside the job created
# suffix:       which string must be added to the input/output/sub files to identify. For example, in ABITEM_opt_r1_HS.log, "opt" is the suffix
# keyword:      a name given to this job. 
# istate:       state from which the information must be extracted 
# fstate:       state where the information will be stored
# hierarchy:    expected order of jobs inside workflow
# requisites:   list of keywords of jobs that should have finished before this one can start
# constrains:   list of keywords of jobs that should have NOT finished before this one can start
# must_be_good: whether the job can considered done even if it did not finish correctly. 
    # if must_be_good = true, it will try hard to do the task. For instance, if the user requested an optimization, it will not stop until reaching the minimum
    # if must_be_good = false, it will perform just one attempt

### Section 4: QC data

In [13]:
## Finally, we have the information of the actual Quantum Chemistry (QC) computation.
qc_data = fill_qc_data(qc_data)
print(qc_data)

## This is the information that will be passed to the QC software to create the input 

## In principle, you can update the information here (eg. you can change the basis set) and re-submit. 
## If you do so, and everything works well, the code will compare the old and new qc_datas and if an update is detected...
## ...it will resubmit even if the same job under the old qc_data was flagged as complete.
## In other words, it should be save to update the qc_data, but I haven't tested all changes.

Formatted input interpretation: ( self -> Instance of class Input() )
---------------------------------------------------
self.Key                 | Data Type           | Value     
---------------------------------------------------
self.software            | <class 'str'>       | g16       
self.jobtype             | <class 'str'>       | opt       
self.functional          | <class 'str'>       | pbe       
self.basis               | <class 'str'>       | def2svp   
self.is_grimme           | <class 'bool'>      | True      
self.loose_opt           | <class 'bool'>      | True      
self.tight_opt           | <class 'bool'>      | False     
self.grimme_type         | <class 'str'>       | d2        



In [14]:
## Currently, the code accepts a very limited set of qc_data options. Only Quantum Espresso and Gaussian codes are implemented, 
## together with a very limited set of functionals, basis sets, etc...

In [15]:
## In Quantum Espresso, we have implemented different Pseudo-Potential Libraries available in the literature. 
## These are called in this section ('pseudo' variable). 
## See below for an example of the qc_data for Quantum Espresso:

#&qc_data
#   software     = 'qe'
#   version      = '7.0'
#   jobtype      = 'vc-relax'
#   functional   = 'pbe'
#   is_hubbard   = True
#   is_grimme    = True
#   grimme_type  = 'd3bj'
#   uterm        = 2.35
#   mix_beta     = 0.7
#   elec_conv    = 1e-5
#   pseudo       = 'vanderbilt'
#/

## 3) Run

In [16]:
## Once the system and a job_file are prepared, we can submit the computation in the appropriate cluster
## To do so:

## 1) Configure the Cluster Environment using the "scope_config" script, installed by default: 

        ## The computation cluster should have Gaussian 16 installed, and a **configured environment**. 
        ## You can give it any name, but for the sake of simplicity, here we will call it "scope_env_cluster.npy" 

## 2) Upload the system file and the test_job.job to the cluster

        ## In principle, you should upload the system file to a dedicated folder inside the env.sys_path of the cluster enviromnent (env). 
        ## So, if env.sys_path = '/home/user/scope/tutorials/data/Tutorial_4/Systems', the system file should go to: '/home/user/scope/tutorials/data/Tutorial_4/Systems/water/water.npy'
        
        ## In practice, you can upload the system file anywhere in the cluster. 
        ## However, the cluster's Environment will force the system to be saved in the expected path: '//home/user/scope/tutorials/data/Tutorial_4/Systems/water/water.npy'
        ## This is the price to pay for the Environment controlling the paths

        ## The job_file can be saved anywhere. 

## 3) Then, you can use the 'scope_run_job' script.

        ## If the paths are the expected ones, you should type: 

```bash
scope_run_job -n scope_env_cluster.npy -s Systems/water/water.npy -j test_job.job
```
```

In [17]:
## The environment of the computation cluster (scope_env_cluster.npy in the example) will automatically update the paths stored in the system, and execute the job defined in test_job.
## If everything worked well, then you should have a gaussian16 job running in your cluster. 
## Whenever it finishes, just execute the same command above. Then, scope will register your job

## 4) Register

In [18]:
## When executing "scope_run_job", the script takes the indicated (in the job_file) Branch of the system, goes through the computational workflow...
## ... and takes the appropriate actions: 
 
## 1) If the job has not been submitted, it creates the input and submission file and does the submission.
## 2) If the job is running, it does nothing
## 3) If the job has finished, it reads the output and decides whether it finished 'well'
##   3.1) If not, it takes appropriate action. For instance, 
##      3.1.1) If it is a GEOMETRY OPTIMIZATION that didn't reach a zero-gradient solution, it creates and submits a new computation that continues the previous run
##      3.1.2) If it is an SCF computation in Quantum Espresso that didn't converge, it will modify some parameters and resubmit.
##      3.1.3) If the computation was aborted by SLURM or for some other reason, the logfile will be discarded, and the computation will be resubmitted.
##   3.2) If the job finished well, the logfile is parsed, and the relevant data is stored in the final state (fstate) defined by the computation
##        The job will be considered done, and even if you resubmit the same command, the computations won't run again. 

In [19]:
## In tutorial number 6, we will see how those computations are "REGISTERED"